# Structured Outputs

Structured outputs are useful when model responses are passed as inputs to other components of a system. 


## OpenAI API Structured Output

Historically, the OpenAI interface offered JSON output. However, using JSON output does not ensure adherence to a schema (data types, for example, may not be enforced). An alternative is to define the output schema using Pydantic.

A useful reference is the entry on [Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs) from the API Documentation.

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
from pydantic import BaseModel
import os
os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

[Pydantic](https://docs.pydantic.dev/latest/) is a data validation library for Python. In Pydantic, we define [Models](https://docs.pydantic.dev/latest/concepts/models/) which are classes which inherit from [`BaseModel`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel) and define [`Fields`](https://docs.pydantic.dev/latest/api/fields/#pydantic.fields.Field) as annotated attributes.

In simpler terms, Pydantic ensures data matches expected data types, and automatically converts them where possible if not. 

In [3]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


In [ ]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": "Extract the event information."},
        #some models may ignore system prompts if it contradicts higher priorities (e.g. helpfulness over roleplaying/speaking in a funny tone. 
        #in that case, change 'role' to 'developer', which has an even higher priority than system and tells the model that the instruction is absolute
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent,
)

event = response.output_parsed

In [5]:
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [ ]:
type(event)
#note event is no longer a string (like content), but of the type (more specifically, user-defined class) CalendarEvent

__main__.CalendarEvent

In [ ]:
#and not just any object; event is a Pydantic object
#why is that significant? that allows you to model_dump the output as a dictionary: 
event.model_dump()
#you can also dump it in JSON or other (structured) formats 

{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}

## LangChain and Pydantic

LangChain offers capabilities to structure outputs following a specific schema. 

In the example below, we use LangChain to obtain a Joke object with a specific structure given by a Pydantic BaseModel.

In [6]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL, 
                      model_provider="openai",
                       base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                       default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
                      )

In [7]:
from typing import Optional
from pydantic import BaseModel, Field

class Joke(BaseModel):
    setup: str=Field(description="The setup of the joke")
    punchline: str=Field(description="The punchline of the joke")
    rating: Optional[int] = Field(
        default=None, description="How funny the joke is, from 1 to 10"
    )

structured_llm = llm.with_structured_output(Joke)

jk = structured_llm.invoke("Tell me a joke about cats")

In [ ]:
jk.model_dump()

{'setup': 'Why was the cat sitting on the computer?',
 'punchline': 'Because it wanted to keep an eye on the mouse!',
 'rating': 8}

## LangChain and TypedDict

We can also use a [typed dictionary or TypedDict](https://typing.python.org/en/latest/spec/typeddict.html) in Python to define the structure of our output. The keyword `Annotated[]` is used to wrap attributes of a typed dictionary.

In [ ]:
from typing import Optional
from typing_extensions import Annotated, TypedDict

class JokeDict(TypedDict):
    setup: Annotated[str, ..., "The setup of the joke"] # No default (that's what the ... stands for), with description
    punchline: Annotated[str, ..., "The punchline of the joke"] # No default, with description
    rating: Annotated[Optional[int], None, "How funny the joke is, from 1 to 10"] # Default of None, with description)

In [10]:
structured_llm_dict = llm.with_structured_output(JokeDict)
jk_dict = structured_llm_dict.invoke("Tell me a joke about dogs")


In [11]:
jk_dict

{'setup': 'Why did the dog sit in the shade?',
 'punchline': "Because he didn't want to become a hot dog!",
 'rating': 7}